# STOR 120: Midterm 1 Table Operations Practice Quiz

This guide mirrors the interactive Jupyter notebook quiz. Use it to review the key coding questions, try writing the answers, and verify your logic against the step-by-step explanations below.

---

### Setup Code & Dataset
To set up this dataset in your own Jupyter notebook, run:

In [3]:
from datascience import Table, are
import numpy as np

skyscrapers = Table().with_columns(
    'name', np.array(['Empire State Building', 'Burj Khalifa', 'Taipei 101', 'One World Trade Center', 'The Shard', 'Willis Tower']),
    'material', np.array(['steel', 'steel/concrete', 'steel/concrete', 'steel/concrete', 'glass/steel', 'steel']),
    'height_ft', np.array([1454, 2717, 1667, 1776, 1016, 1450]),
    'city', np.array(['New York City', 'Dubai', 'Taipei', 'New York City', 'London', 'Chicago']),
    'completed_year', np.array([1931, 2010, 2004, 2014, 2012, 1974])
)
skyscrapers

name,material,height_ft,city,completed_year
Empire State Building,steel,1454,New York City,1931
Burj Khalifa,steel/concrete,2717,Dubai,2010
Taipei 101,steel/concrete,1667,Taipei,2004
One World Trade Center,steel/concrete,1776,New York City,2014
The Shard,glass/steel,1016,London,2012
Willis Tower,steel,1450,Chicago,1974


---

## Question 1: Selecting and Sorting
**Task:** Write a line of code that creates a table called `tall_skyscrapers` containing only the columns `name` and `height_ft`, sorted from **tallest to shortest**.

In [7]:
tall_skyscrapers = skyscrapers.select("name", "height_ft").sort("height_ft", descending=True)
tall_skyscrapers

name,height_ft
Burj Khalifa,2717
One World Trade Center,1776
Taipei 101,1667
Empire State Building,1454
Willis Tower,1450
The Shard,1016


In [8]:
# Solution:
tall_skyscrapers = skyscrapers.select('name', 'height_ft').sort('height_ft', descending=True)
tall_skyscrapers

name,height_ft
Burj Khalifa,2717
One World Trade Center,1776
Taipei 101,1667
Empire State Building,1454
Willis Tower,1450
The Shard,1016


### Explanation:
1. We use `.select('name', 'height_ft')` to keep only those two columns. (Using `.drop('material', 'city', 'completed_year')` would also work, but `.select()` is much more direct).
2. We chain `.sort('height_ft', descending=True)` to sort the table. By default, `.sort()` sorts in ascending order (smallest to largest), so setting `descending=True` is required to list the tallest building first.

---

## Question 2: Table Predicates & Filtering
**Task:** Filter the `skyscrapers` table to find all buildings located in **New York City** that were completed **after the year 2000**. Store the resulting table in `modern_nyc`.

In [12]:
modern_nyc = skyscrapers.where("city", are.containing("New York City")).where("completed_year", are.above(2000))
# Using .containing is not as strict as .equal_to
modern_nyc

name,material,height_ft,city,completed_year
One World Trade Center,steel/concrete,1776,New York City,2014


In [13]:
# Solution:
modern_nyc = skyscrapers.where('city', are.equal_to('New York City')).where('completed_year', are.above(2000))
modern_nyc

name,material,height_ft,city,completed_year
One World Trade Center,steel/concrete,1776,New York City,2014


### Explanation:
1. We use `.where('city', are.equal_to('New York City'))` to filter rows where the city is NYC.
2. We chain a second `.where('completed_year', are.above(2000))` to apply the second condition.
3. Remember that each `.where()` call returns a *new* table, so chaining them sequentially acts as an **AND** condition.

---

## Question 3: Table vs. Array Extraction (CRITICAL CONCEPT!)
**Task:** Calculate the **average height** (in feet) of all the skyscrapers in our dataset. Store this single numerical value in `avg_height`.

In [14]:
avg_height = np.average(skyscrapers.column("height_ft"))
avg_height

1680.0

In [15]:
# Solution:
avg_height = np.mean(skyscrapers.column('height_ft'))
avg_height

1680.0

### Explanation:
This is a major midterm concept!
* `.select('height_ft')` returns a **Table** containing one column. You cannot perform arithmetic functions like `np.mean()` on a Table.
* `.column('height_ft')` extracts the actual data as a **NumPy array** of numbers. You can perform arithmetic aggregations directly on arrays.
* Therefore, we extract the array first, then pass it to `np.mean()` to calculate the single average value: `1780.0`.

---

## Question 4: Advanced Manipulation (Take vs. Where)
**Task:** Identify the **oldest skyscraper** in our dataset and extract its **name** as a string. Store it in `oldest_building_name`.

In [19]:
oldest_building_name = skyscrapers.sort("completed_year", descending=False).column("name").item(0)
oldest_building_name

'Empire State Building'

In [20]:
# Solution:
oldest_building_name = skyscrapers.sort('completed_year').column('name').item(0)
oldest_building_name

'Empire State Building'

### Explanation:
1. We sort the table by `completed_year` in ascending order (default), which puts the oldest building in the very first row.
2. We extract the `name` column as an array using `.column('name')`.
3. We use `.item(0)` to extract the first element of that array (the string `'Empire State Building'`).

*Alternative Method (using `.take()`):*

In [21]:
oldest_building_name_alt = skyscrapers.sort('completed_year').take(0).column('name').item(0)
oldest_building_name_alt

'Empire State Building'

* `.take(0)` grabs the first row as a Table.
* Then `.column('name')` grabs that single element as an array, and `.item(0)` extracts the string.